In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


In [6]:
data = pd.read_csv('data/data_dl.csv')
features = data.columns[:-1]

In [8]:
len(features)

44

In [ ]:
data['Target'].unique()


array(['Standard', 'Good', 'Poor'], dtype=object)

In [14]:
class MLPClassifier(nn.Module):
    def __init__(self, cat_dims, num_cols, hidden = [128, 64, 32], dropout = 0.3):
        super().__init__()
        self.n_cat = len(cat_dims)
        self.n_nums = len(num_cols)
        self.embs = nn.ModuleList([nn.Embedding(v, d) for v, d in cat_dims])
        in_d = sum(d for _, d in cat_dims) + len(num_cols)
        layers = []
        for h in hidden:
            layers += [nn.Linear(in_d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_d = h
        layers.append(nn.Linear(in_d, 3))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x):
        cx = x[:, :self.n_cat].long()          # integer-encoded categoricals
        nx = x[:, self.n_cat:].float()
        x = torch.cat([e(cx[:, i]) for i, e in enumerate(self.embs)] + [nx], dim=1)
        return self.net(x)